In [ ]:
REPO_URL = "https://github.com/huyvanzzz/finetune-InternVL2.git"
TARGET_BRANCH = "feature/trajectory-pretrain-qformer-concat-bestshot-bf16"
PROJECT_DIR = "/kaggle/working/finetune-InternVL2"
CONFIG_PATH = "internvl_config_traj_concat_bestshot_bf16_2gpu.yaml"


In [ ]:
import os, subprocess, pathlib
if not pathlib.Path(PROJECT_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
subprocess.run(["git", "fetch", "origin", TARGET_BRANCH], check=True)
subprocess.run(["git", "checkout", "-B", TARGET_BRANCH, f"origin/{TARGET_BRANCH}"], check=True)
print("Current repo:", os.getcwd())
print("Current branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("Current commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
!pip uninstall -y torch torchvision torchaudio triton flash-attn torchao peft || true
!pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu126 torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1
!pip install -q --no-cache-dir bitsandbytes transformers==4.46.2 datasets accelerate timm evaluate rouge_score scikit-learn safetensors huggingface_hub sentencepiece mlflow pyyaml pillow tqdm peft==0.18.1


In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda build:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('capability:', torch.cuda.get_device_capability(0))
    x = torch.zeros(1, device='cuda', dtype=torch.float16)
    print('cuda fp16 tensor ok:', x.dtype)
!python -m py_compile train.py wad_dataset.py qformer_bridge.py scripts/test_infer.py scripts/prepare_qformer.py scripts/smoke_qformer_bridge.py


In [ ]:
import subprocess
subprocess.run(["python", "build_frame_index.py"], input="n\n", text=True, check=True)


In [ ]:
!python scripts/prepare_qformer.py --config {CONFIG_PATH}


In [ ]:
!python scripts/smoke_qformer_bridge.py --config {CONFIG_PATH}


In [ ]:
import subprocess

TRAIN_CHECKPOINT = ""  # resume a finetune run from a local folder or HF repo id
PRETRAIN_CHECKPOINT = ""  # preload qformer bridge + trajectory branch from pretrain checkpoint
TRAIN_START_EPOCH = None  # only for finetune resume via TRAIN_CHECKPOINT
TRAIN_START_STEP = None   # only for finetune resume via TRAIN_CHECKPOINT
cmd = ["accelerate", "launch", "--num_processes", "2", "train.py", "--config", CONFIG_PATH]
if TRAIN_CHECKPOINT:
    cmd += ["--checkpoint", TRAIN_CHECKPOINT]
if PRETRAIN_CHECKPOINT:
    cmd += ["--pretrain_checkpoint", PRETRAIN_CHECKPOINT]
if TRAIN_START_EPOCH is not None:
    cmd += ["--start_epoch", str(TRAIN_START_EPOCH)]
if TRAIN_START_STEP is not None:
    cmd += ["--start_step", str(TRAIN_START_STEP)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import subprocess
import yaml
import json
from pathlib import Path

with open(CONFIG_PATH) as f:
    runtime_cfg = yaml.safe_load(f)
BASE_OUTPUT_DIR = Path(runtime_cfg["training"]["output_dir"])
RESULT_STEM = Path(CONFIG_PATH).stem.replace("internvl_config_", "")

CHECKPOINT_DIR = ""  # optional: fill manually with outputs/.../epoch_x
EVAL_ALL_EPOCHS = True

if CHECKPOINT_DIR:
    checkpoints_to_eval = [Path(CHECKPOINT_DIR)]
else:
    run_dirs = sorted([p for p in BASE_OUTPUT_DIR.glob('*') if p.is_dir()], key=lambda p: p.stat().st_mtime)
    latest_run_dir = run_dirs[-1] if run_dirs else None
    if latest_run_dir is None:
        checkpoints_to_eval = []
    elif EVAL_ALL_EPOCHS:
        checkpoints_to_eval = sorted(
            [p for p in latest_run_dir.glob('epoch_*') if p.is_dir() and p.name.split('_')[-1].isdigit()],
            key=lambda p: int(p.name.split('_')[-1]),
        )
    else:
        epoch_dirs = sorted(
            [p for p in latest_run_dir.glob('epoch_*') if p.is_dir() and p.name.split('_')[-1].isdigit()],
            key=lambda p: int(p.name.split('_')[-1]),
        )
        checkpoints_to_eval = [epoch_dirs[-1]] if epoch_dirs else []

if not checkpoints_to_eval:
    print("No checkpoint found. Train first, then rerun this cell.")
else:
    for checkpoint_dir in checkpoints_to_eval:
        epoch_suffix = checkpoint_dir.name
        output_file = f"results/{RESULT_STEM}_eval_test_alter_{epoch_suffix}.json"
        print(f"\n=== Evaluating {checkpoint_dir} ===")
        subprocess.run([
            "python", "scripts/test_infer.py",
            "--config", CONFIG_PATH,
            "--checkpoint", str(checkpoint_dir),
            "--split", "test_alter",
            "--output_file", output_file,
        ], check=True)
        with open(output_file, "r", encoding="utf-8") as f:
            result_payload = json.load(f)
        print("Metrics:", json.dumps(result_payload.get("metrics", {}), ensure_ascii=False, indent=2))
        print("Result JSON:", output_file)
        print("Pairs JSON:", output_file.replace('.json', '_pairs.json'))



In [ ]:
import subprocess
subprocess.run(["python", "scripts/visualize_training.py", "--save", "training_plot.png"], check=True)

from IPython.display import Image, display
display(Image("training_plot.png"))


In [ ]:
import json
import yaml
from pathlib import Path

with open(CONFIG_PATH) as f:
    runtime_cfg = yaml.safe_load(f)
base = Path(runtime_cfg["training"]["output_dir"])
candidates = sorted(base.glob("*/metrics.json"), key=lambda p: p.stat().st_mtime, reverse=True)
METRICS_PATH = str(candidates[0])

with open(METRICS_PATH) as f:
    m = json.load(f)

print("=== EPOCH SUMMARY ===")
for ep in m["epoch_summary"]:
    print(f"  Epoch {ep['epoch']}: avg_train_loss = {ep['avg_train_loss']:.6f}")

print("\n=== VALIDATION LOSS ===")
for v in m["val_loss"]:
    print(f"  Epoch {v['epoch']} | Step {v['step']:>6}: {v['loss']:.6f}")
if m["val_loss"]:
    best = min(m["val_loss"], key=lambda x: x["loss"])
    print(f"\n  Best: {best['loss']:.6f} @ step {best['step']}")
